In [ ]:
import scanpy as sc
import pandas as pd

In [ ]:
adata = sc.read_h5ad("raw/SrivatsanTrapnell2020_sciplex3.h5ad")

In [ ]:
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata,n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]

In [ ]:
adata.write_h5ad(f"raw/Sciplex3_hvg2000.h5ad")

In [ ]:
adata

In [ ]:
import numpy as np
adata_sample = adata[np.random.choice(adata.n_obs, int(adata.n_obs * 0.01), replace=False)].copy()

In [ ]:
adata_sample.write_h5ad("raw/Sciplex3_test.h5ad")

In [ ]:
adata = sc.read_h5ad("raw/Sciplex3_hvg2000.h5ad")

In [ ]:
adata = adata[adata.obs[["time", "perturbation","dose_value"]].notna().all(axis=1), :].copy() #部分time是nan

In [ ]:
adata.obs["perturbation"] = adata.obs["perturbation"].astype(str).str.strip()

In [ ]:
adata_24 = adata[adata.obs["time"]==24].copy()

In [ ]:
adata_24

In [ ]:
adata_24.write_h5ad(f"raw/Sciplex3_24h_hvg2000.h5ad")

In [ ]:
adata_72 = adata[adata.obs["time"]==72].copy()

In [ ]:
adata_72

In [ ]:
adata_72.write_h5ad(f"raw/Sciplex3_72h_hvg2000.h5ad")

In [ ]:
adata.obs["time"].value_counts()

In [ ]:
adata

In [ ]:
adata.obs["time"].value_counts()

In [ ]:
import pandas as pd

obs = adata_24.obs.copy()

# 定义唯一孔：plate + well
obs["plate_well"] = obs["plate"].astype(str) + "_" + obs["well"].astype(str)

# 统计每个 (dose_value, perturbation, time) 对应多少个唯一孔
condition_well_counts = (
    obs.groupby(["dose_value", "perturbation", "time"])["plate_well"]
    .nunique()
    .reset_index(name="n_wells")
    .sort_values(["dose_value", "perturbation", "time"])
)

condition_well_counts = condition_well_counts[condition_well_counts["n_wells"] != 0]

print(condition_well_counts)


In [ ]:
adata_24.obs["perturbation"].value_counts()

In [ ]:
adata_72.obs["perturbation"].value_counts()

In [ ]:
obs = adata_24.obs.copy()
obs["plate_well"] = obs["plate"].astype(str) + "_" + obs["well"].astype(str)

wells_per_perturb = (
    obs.groupby("perturbation", observed=True)["plate_well"]
    .nunique()
    .sort_values(ascending=False)
)

ratio = wells_per_perturb["control"] / wells_per_perturb[wells_per_perturb.index != "control"].iloc[0]
print(ratio)


In [ ]:
adata.obs["well"].value_counts()

In [ ]:
adata.obs["cell_line"].value_counts()

In [ ]:
adata.obs['dose_value'].value_counts()

In [ ]:
adata.obs["perturbation"].value_counts()

In [ ]:
df = pd.DataFrame(
    adata.obs["perturbation"].unique(),
    columns=["drug"]
)
df.to_csv("target_drugs_sciplex3.csv", index=False)

In [ ]:
adata.obs["time"].value_counts()